[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/18_3d_geometry_and_rendering.ipynb)

# 18. 3D geometry and rendering

3D point를 camera로 투영하는 기본 기하에서 ray sampling, NeRF volume rendering, Gaussian splatting의 핵심 tensor 계산까지 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Rigid transform

homogeneous coordinate로 point를 camera 좌표계로 옮긴다.


In [ ]:
points = torch.tensor([[1., 0., 2.], [0., 1., 4.]], device=device)
R = torch.eye(3, device=device)
t = torch.tensor([0.1, 0.2, 0.0], device=device)

camera = points @ R.T + t
print(camera)


In [ ]:
_ = profile_call("rigid transform", lambda p: p@R.T+t, points)


## 2. Perspective projection

x/z, y/z와 camera intrinsic을 적용한다.


In [ ]:
fx = fy = 100.0
cx = cy = 32.0

u = fx * camera[:, 0] / camera[:, 2] + cx
v = fy * camera[:, 1] / camera[:, 2] + cy

pixels = torch.stack([u, v], dim=-1)
print(pixels)


In [ ]:
def project_points(p):
    u_ = fx * p[:, 0] / p[:, 2] + cx
    v_ = fy * p[:, 1] / p[:, 2] + cy
    return torch.stack([u_, v_], dim=-1)

_ = profile_call("perspective projection", project_points, camera)


## 3. Ray sampling

origin + depth * direction으로 3D sample point를 만든다.


In [ ]:
origin = torch.tensor([0., 0., 0.], device=device)
direction = F.normalize(torch.tensor([1., 0.5, 2.], device=device), dim=0)
depths = torch.linspace(0.5, 2.0, 5, device=device)

samples = origin + depths[:, None] * direction
print(samples)


In [ ]:
_ = profile_call("ray samples", lambda: origin + depths[:,None]*direction)


## 4. NeRF volume rendering

density에서 alpha와 transmittance를 만들고 RGB를 weighted sum한다.


In [ ]:
sigma = torch.tensor([0.2, 0.5, 1.0, 0.3], device=device)
rgb = torch.tensor(
    [[1.,0.,0.], [0.,1.,0.], [0.,0.,1.], [1.,1.,1.]],
    device=device,
)
delta = torch.full((4,), 0.5, device=device)

alpha = 1 - torch.exp(-sigma * delta)
trans = torch.cumprod(torch.cat([torch.ones(1,device=device), 1-alpha+1e-8])[:-1], dim=0)
weights = trans * alpha
pixel = (weights[:,None] * rgb).sum(0)

print("weights:", weights)
print("pixel:", pixel)


In [ ]:
def volume_render_once():
    alpha_ = 1 - torch.exp(-sigma * delta)
    survival_ = torch.cat(
        [torch.ones(1, device=device), 1 - alpha_ + 1e-8]
    )
    trans_ = torch.cumprod(survival_[:-1], dim=0)
    weights_ = trans_ * alpha_
    return (weights_[:, None] * rgb).sum(dim=0)

_ = profile_call("volume rendering", volume_render_once)


## 5. 3D Gaussian covariance

scale과 rotation으로 covariance를 만든다.


In [ ]:
scale = torch.tensor([0.2, 0.1, 0.3], device=device)
S = torch.diag(scale.square())
R = torch.eye(3, device=device)
cov = R @ S @ R.T

print(cov)


In [ ]:
_ = profile_call("Gaussian covariance", lambda: R@S@R.T)


## 6. Alpha compositing

front-to-back Gaussian alpha blending의 최소 형태.


In [ ]:
alpha = torch.tensor([0.2, 0.5, 0.4], device=device)
color = torch.tensor([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]], device=device)

T = torch.cumprod(torch.cat([torch.ones(1,device=device), 1-alpha+1e-8])[:-1], 0)
weights = T * alpha
pixel = (weights[:,None] * color).sum(0)

print("weights:", weights)
print("pixel:", pixel)


In [ ]:
def alpha_composite_once():
    survival_ = torch.cat(
        [torch.ones(1, device=device), 1 - alpha + 1e-8]
    )
    trans_ = torch.cumprod(survival_[:-1], dim=0)
    weights_ = trans_ * alpha
    return (weights_[:, None] * color).sum(dim=0)

_ = profile_call("alpha compositing", alpha_composite_once)


## References and provenance

**[18.1] NeRF**
- 출처: Mildenhall et al., NeRF
- 이 노트북에서 가져온 부분: ray sampling and volume rendering

**[18.2] 3D Gaussian Splatting**
- 출처: Kerbl et al., 3D Gaussian Splatting
- 이 노트북에서 가져온 부분: anisotropic Gaussian covariance and alpha compositing

**[18.3] Hunyuan3D / TRELLIS families**
- 출처: Tencent Hunyuan3D and Microsoft TRELLIS open models
- 이 노트북에서 가져온 부분: modern 3D latent/geometry generation reference families
